<a href="https://colab.research.google.com/github/romikasolanki/https-github.com-romikasolanki-EDA_Optimising_NYC_Taxis_RoEDA_Optimising_NYC_Taxis_Romika-Solanki/blob/main/capstone_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ElecKart Market Mix Modeling Analysis


ElecKart is an e-commerce marketplace based in Ontario, Canada, specializing in electronic products such as cameras, home audio systems, and gaming accessories. It operates on a marketplace model, connecting multiple sellers with customers through its online platform.

## INTRODUCTION
Over the past year, ElecKart has experienced a decline in revenue despite significant investments in marketing and promotional campaigns. The platform is also facing challenges such as low customer engagement, reduced order conversion rates, and high customer churn.

Customers spend very little time on the website and often abandon their carts without completing purchases, indicating issues in user experience and platform efficiency.

This project aims to perform a comprehensive data-driven analysis to identify the key factors affecting ElecKart’s performance. The analysis leverages order-level data, marketing spend, customer feedback (NPS), and external factors such as climate conditions to uncover the root causes behind declining revenue and engagement.

The study focuses on building Market Mix Models (MMM) at a weekly level for three key product subcategories: Camera Accessory, Home Audio, and Gaming Accessory. The objective is to generate actionable insights that can help improve customer experience, optimize marketing spend, and enhance overall business profitability.



## PROBLEM STATEMENT
ElecKart is experiencing a decline in revenue despite significant investments in marketing and promotional activities. The company is facing multiple challenges, including low customer engagement, reduced order conversion rates, and high customer churn.

Customers spend minimal time on the platform and often leave without completing purchases due to a confusing website interface and complex checkout process. Additionally, high cart abandonment is leading to poor inventory utilization and lower turnover rates.

These issues are impacting overall business performance, making it difficult for the company to achieve sustainable growth and profitability.

## Business Objectives

The primary objectives of this analysis are:

- To identify the key factors driving revenue decline and low conversion rates  
- To analyze customer behavior and engagement patterns  
- To evaluate the effectiveness of marketing spend across different channels  
- To understand the impact of external factors such as climate and seasonality on sales  
- To build Market Mix Models (MMM) for key product subcategories to quantify the impact of various drivers on revenue  
- To provide actionable recommendations to improve customer experience, optimize marketing spend, and enhance overall profitability


## DATA UNDERSTANDING
The analysis uses multiple datasets to evaluate ElecKart’s performance from July 2015 to June 2016.

The primary dataset contains order-level information such as order date, revenue (GMV), units sold, delivery timelines, payment type, customer ID, and product categories.

In addition to this, supporting datasets include:
- Marketing spend across different channels
- Monthly Net Promoter Score (NPS) as a measure of customer satisfaction
- Special sale events and promotional periods
- Stock index data representing company performance
- Climate data for Ontario (2015–2016) to capture seasonal effects

These datasets will be combined to perform a comprehensive analysis and build Market Mix Models (MMM) at a weekly level for selected product subcategories.

## DATA PREPARATION

integrated cloud-based storage with my analysis environment to ensure seamless and scalable data access.”





In [ ]:
#Mounting Drive
from google.colab import drive
drive.mount('/content/gdrive')


In [ ]:
#LOADING DATAESETS
# Import libraries
import pandas as pd
import numpy as np

# Load main dataset
orders = pd.read_csv('/content/gdrive/MyDrive/capstone project/ConsumerElectronics.csv')

# Load marketing + NPS data
media = pd.read_excel('/content/gdrive/MyDrive/capstone project/Media data and other information.xlsx',
                      sheet_name='Media Investment')
# Load climate data
climate_2015 = pd.read_excel('/content/gdrive/MyDrive/capstone project/ONTARIO-2015.xlsx',header=2)
climate_2016 = pd.read_excel('/content/gdrive/MyDrive/capstone project/ONTARIO-2016.xlsx',header=2)


In [ ]:
#oeders data info
orders.head()
orders.shape
orders.info()



In [ ]:
orders['gmv'] = pd.to_numeric(orders['gmv'], errors='coerce')
orders['deliverybdays'] = pd.to_numeric(orders['deliverybdays'], errors='coerce')
orders['deliverycdays'] = pd.to_numeric(orders['deliverycdays'], errors='coerce')

orders['order_date'] = pd.to_datetime(orders['order_date'])

In [ ]:
orders.isnull().sum()

In [ ]:
orders = orders.dropna(subset=['gmv'])

In [ ]:
delivery_data = orders.dropna(subset=['deliverybdays', 'deliverycdays'])

In [ ]:
orders = orders[(orders['order_date'] >= '2015-07-01') &
                (orders['order_date'] <= '2016-06-30')]

In [ ]:
filtered_data = orders[
    orders['product_analytic_sub_category'].isin([
        'CameraAccessory',
        'HomeAudio',
        'GamingAccessory'
    ])
]

In [ ]:
filtered_data['order_date'] = pd.to_datetime(filtered_data['order_date'])

filtered_data['Year'] = filtered_data['order_date'].dt.year
filtered_data['Month'] = filtered_data['order_date'].dt.month
filtered_data['Week'] = filtered_data['order_date'].dt.isocalendar().week
filtered_data['Day'] = filtered_data['order_date'].dt.day

In [ ]:
filtered_data = filtered_data[filtered_data['gmv'].notnull()]

In [ ]:
filtered_data.columns

In [ ]:
filtered_data.drop(columns=['week'], inplace=True)

In [ ]:
filtered_data['week'] = filtered_data['order_date'].dt.to_period('W')

In [ ]:
filtered_data['is_payday'] = filtered_data['Day'].isin([1, 15]).astype(int)

In [ ]:
weekly_data = filtered_data.groupby(
    ['week', 'product_analytic_sub_category']
).agg({
    'gmv': 'sum',
    'units': 'sum',
    'is_payday': 'sum'
}).reset_index()

In [ ]:
weekly_data.head()
weekly_data.info()

In [ ]:
weekly_pivot = weekly_data.pivot(
    index='week',
    columns='product_analytic_sub_category',
    values='gmv'
).reset_index()

In [ ]:
weekly_pivot.head()
weekly_pivot.shape

In [ ]:
orders['product_analytic_sub_category'].unique()

In [ ]:
filtered_data.shape
filtered_data['product_analytic_sub_category'].value_counts()


In [ ]:
#media data info
media.head()
media.shape
media.info()

In [ ]:
media = pd.read_excel('/content/gdrive/MyDrive/capstone project/Media data and other information.xlsx',
                      sheet_name='Media Investment', header=2)  # try changing header row

In [ ]:
media = media.loc[:, ~media.columns.str.contains('Unnamed')]

In [ ]:
media['Date'] = pd.to_datetime(media[['Year', 'Month']].assign(DAY=1))

In [ ]:
for col in media.columns:
    if col not in ['Year', 'Month', 'Date']:
        media[col] = pd.to_numeric(media[col], errors='coerce')

In [ ]:
media.describe()

In [ ]:
media.isnull().sum()

In [ ]:
media = media.drop(columns=['Radio', 'Other'])

In [ ]:
media.fillna(0, inplace=True)

In [ ]:
media = media.dropna(subset=['Year', 'Month'])

In [ ]:
media.head()
media.shape
media.info()

In [ ]:
climate_2015 = pd.read_excel('/content/gdrive/MyDrive/capstone project/ONTARIO-2015.xlsx', header=24)
climate_2016 = pd.read_excel('/content/gdrive/MyDrive/capstone project/ONTARIO-2016.xlsx', header=24)

In [ ]:
climate_2015.columns
climate_2016.columns

In [ ]:
climate_2015['Date'] = pd.to_datetime(climate_2015['Date/Time'])
climate_2016['Date'] = pd.to_datetime(climate_2016['Date/Time'])

In [ ]:
climate = pd.concat([climate_2015, climate_2016])

In [ ]:
climate['week'] = climate['Date'].dt.to_period('W').apply(lambda r: r.start_time)

climate_weekly = climate.groupby('week')['Mean Temp (°C)'].mean().reset_index()

In [ ]:
weekly_data = filtered_data.groupby(
    ['week', 'product_analytic_sub_category']
).agg({
    'gmv': 'sum',
    'units': 'sum'
}).reset_index()

In [ ]:
weekly_data = weekly_data.merge(climate_weekly, on='week', how='left')

In [ ]:
weekly_data.head()
weekly_data.info()

## FEATURE ENGINEERING

“Feature engineering was performed to derive meaningful variables such as delivery delay, order value, and delivery status to better understand operational inefficiencies and customer experience.”



In [ ]:
# Convert week to actual date
weekly_pivot['week_start'] = weekly_pivot['week'].dt.start_time

# Extract day
weekly_pivot['day'] = weekly_pivot['week_start'].dt.day

# Payday flag
weekly_pivot['is_payday'] = weekly_pivot['day'].isin([1, 15]).astype(int)

In [ ]:
# Example holiday list (can expand later)
holidays = [
    '2022-01-01',  # New Year
    '2022-07-01',  # Canada Day
    '2022-12-25'   # Christmas
]

holidays = pd.to_datetime(holidays)

weekly_pivot['is_holiday'] = weekly_pivot['week_start'].isin(holidays).astype(int)

In [ ]:
import holidays

canada_holidays = holidays.CA()

weekly_pivot['is_holiday'] = weekly_pivot['week_start'].apply(
    lambda x: 1 if x in canada_holidays else 0
)

In [ ]:
climate.columns = climate.columns.str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')

In [ ]:
climate_weekly.head()
weekly_pivot.head()

In [ ]:
climate.columns

In [ ]:
# Step 1: Clean column names
climate.columns = climate.columns.str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')

# Step 2: Convert date
climate['date/time'] = pd.to_datetime(climate['date/time'])

# Step 3: Create week column
climate['week'] = climate['date/time'].dt.to_period('W')

# Step 4: Aggregate
climate_weekly = climate.groupby('week').agg({
    'mean_temp_°c': 'mean',
    'total_rain_mm': 'sum'
}).reset_index()

In [ ]:
climate['date/time'] = pd.to_datetime(climate['date/time'])
climate['week'] = climate['date/time'].dt.to_period('W')

In [ ]:
climate_weekly = climate.groupby('week').agg({
    'mean_temp_°c': 'mean',
    'total_rain_mm': 'sum'
}).reset_index()

In [ ]:
weekly_pivot = weekly_pivot.merge(climate_weekly, on='week', how='left')

In [ ]:
weekly_pivot['temp_category'] = pd.cut(
    weekly_pivot['mean_temp_°c'],
    bins=[-10, 10, 20, 30, 50],
    labels=['Cold', 'Mild', 'Warm', 'Hot']
)

In [ ]:
climate.columns

In [ ]:
weekly_pivot.columns

In [ ]:
weekly_pivot['is_rainy'] = (weekly_pivot['total_rain_mm'] > 0).astype(int)

In [ ]:
weekly_pivot['is_rainy'] = (weekly_pivot['total_rain_mm'] > 0).astype(int)

weekly_pivot['heavy_rain'] = (weekly_pivot['total_rain_mm'] > 50).astype(int)

weekly_pivot['extreme_temp'] = (
    (weekly_pivot['mean_temp_°c'] < 0) |
    (weekly_pivot['mean_temp_°c'] > 30)
).astype(int)

In [ ]:
#LAG FEATURE
weekly_data['gmv_lag1'] = weekly_data['gmv'].shift(1)

In [ ]:
#ROLLIMG MEAN
weekly_data['gmv_roll_mean'] = weekly_data['gmv'].rolling(4).mean()

In [ ]:
#GROWTH
weekly_data['gmv_roll_mean'] = weekly_data['gmv'].rolling(4).mean()

In [ ]:
weekly_data.head()
weekly_data.info()

## EDA

In [ ]:
import matplotlib.pyplot as plt

weekly_pivot.shape
weekly_pivot.info()
weekly_pivot.describe()
weekly_pivot.isnull().sum().sort_values(ascending=False)
weekly_pivot[['CameraAccessory', 'GamingAccessory', 'HomeAudio']].hist(figsize=(12,4))
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Shape of dataset:", weekly_pivot.shape)

print("\nColumns:")
print(weekly_pivot.columns.tolist())

print("\nData Types:")
print(weekly_pivot.dtypes)

print("\nFirst 5 rows:")
display(weekly_pivot.head())

# Missing values summary
missing_summary = pd.DataFrame({
    'Missing_Count': weekly_pivot.isnull().sum(),
    'Missing_Percent': (weekly_pivot.isnull().sum() / len(weekly_pivot)) * 100
}).sort_values(by='Missing_Count', ascending=False)

print("\nMissing Values Summary:")
display(missing_summary)

# Duplicate rows
print("\nDuplicate rows:", weekly_pivot.duplicated().sum())

# Duplicate weeks
if 'week' in weekly_pivot.columns:
    print("Duplicate week values:", weekly_pivot['week'].duplicated().sum())

# Basic summary stats
print("\nSummary Statistics:")
display(weekly_pivot.describe(include='all'))

The pre-analytical weekly dataset was checked for structure and quality before performing EDA. The dataset contains 53 weekly observations with target, business-calendar, and climate variables. Data types were appropriate for analysis, and no duplicate rows or duplicate weekly records were found. Only a small number of missing values were observed in CameraAccessory, HomeAudio, mean_temp_°c, and temp_category, while the remaining variables were complete. Overall, the dataset was considered suitable for further exploratory analysis.


In [ ]:
#UNIVARIATE ANALYSIS
import matplotlib.pyplot as plt

target_cols = ['CameraAccessory', 'GamingAccessory', 'HomeAudio']

# Summary table
target_summary = weekly_pivot[target_cols].describe().T
target_summary['median'] = weekly_pivot[target_cols].median()
target_summary['missing_values'] = weekly_pivot[target_cols].isnull().sum()
target_summary['skewness'] = weekly_pivot[target_cols].skew()

print("Target Variable Summary:")
display(target_summary)

# Histograms
weekly_pivot[target_cols].hist(figsize=(14, 8), bins=10)
plt.suptitle("Distribution of Target Variables", fontsize=14)
plt.tight_layout()
plt.show()

# Boxplot
plt.figure(figsize=(10, 6))
weekly_pivot[target_cols].boxplot()
plt.title("Boxplot of Target Variables")
plt.ylabel("Weekly Revenue")
plt.show()


Insight

CameraAccessory shows relatively high weekly revenue with moderate variation. Its distribution is only slightly right-skewed, suggesting that although most weeks are centered around a stable middle range, a few strong-performing weeks pushed revenue upward.

Interpretation
Mean and median are almost the same, so this category is quite well centered.
Standard deviation is lower than the other two categories, meaning it is more stable.
Slight negative skewness means the distribution is a little tilted toward lower values, but not strongly.
Compared to CameraAccessory and HomeAudio, this category appears less volatile.
Insight

GamingAccessory has the lowest average weekly revenue among the three categories and also shows the most stable distribution. The near-equal mean and median suggest a balanced pattern with fewer extreme high-revenue weeks.
Interpretation
Mean is clearly greater than median.
Skewness is strongly positive.
Maximum value is extremely high compared to the median.
Standard deviation is the highest among all categories.

This means:

most weeks are in a moderate range
a few weeks had very large spikes
HomeAudio is the most volatile category
Insight

HomeAudio displays the strongest right skew and the highest variability, indicating that its revenue is heavily influenced by a few exceptionally high-performing weeks. This suggests stronger peak effects and possibly higher sensitivity to seasonality, special events, or external demand drivers.


In [ ]:
#trend analysis
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

for col in ['CameraAccessory', 'GamingAccessory', 'HomeAudio']:
    plt.plot(weekly_pivot['week_start'], weekly_pivot[col], marker='o', label=col)

plt.title("Weekly Revenue Trend by Category")
plt.xlabel("Week")
plt.ylabel("Revenue")
plt.legend()
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

Time-series trend analysis was performed on the three target categories to understand weekly revenue movement over time. The line chart shows that category sales are not stable across weeks and exhibit clear peaks and dips. CameraAccessory and HomeAudio generally contribute higher weekly revenue, while GamingAccessory remains comparatively lower and more stable.

In [ ]:
#Univariate analysis of business and climate drivers
numeric_features = [
    'is_payday', 'is_holiday',
    'mean_temp_°c', 'total_rain_mm',
    'is_rainy', 'heavy_rain', 'extreme_temp'
]

# Summary statistics
feature_summary = weekly_pivot[numeric_features].describe().T
feature_summary['missing_values'] = weekly_pivot[numeric_features].isnull().sum()

print("Business and Climate Feature Summary:")
display(feature_summary)

# Histograms for numeric features
weekly_pivot[numeric_features].hist(figsize=(16, 10), bins=10)
plt.suptitle("Distribution of Business and Climate Variables", fontsize=14)
plt.tight_layout()
plt.show()

# Frequency table for temp_category
print("Temperature Category Frequency:")
display(weekly_pivot['temp_category'].value_counts(dropna=False))

# Bar plot for temp_category
weekly_pivot['temp_category'].value_counts(dropna=False).plot(kind='bar', figsize=(8,5))
plt.title("Frequency of Temperature Categories")
plt.xlabel("Temperature Category")
plt.ylabel("Count of Weeks")
plt.xticks(rotation=0)
plt.show()

In [ ]:
weekly_pivot[['week', 'week_start', 'is_holiday']].head(15)

In [ ]:
weekly_pivot['is_holiday'].value_counts(dropna=False)

In [ ]:
#media
media_cols = [
    'Total Investment', 'TV', 'Digital', 'Sponsorship',
    'Content Marketing', 'Online marketing', ' Affiliates', 'SEM'
]

media_summary = media[media_cols].describe().T
media_summary['missing_values'] = media[media_cols].isnull().sum()
media_summary['skewness'] = media[media_cols].skew()

print("Media Variable Summary:")
display(media_summary)

# Histograms
media[media_cols].hist(figsize=(16, 10), bins=10)
plt.suptitle("Distribution of Media Variables", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
media_cols = [
    'TV', 'Digital', 'Sponsorship',
    'Content Marketing', 'Online marketing',
    ' Affiliates', 'SEM'
]

# Total spend by channel
channel_totals = media[media_cols].sum().sort_values(ascending=False)

# Share of total spend
channel_share = (channel_totals / channel_totals.sum()) * 100

media_kpi = pd.DataFrame({
    'Total Spend': channel_totals,
    'Spend Share %': channel_share
}).round(2)

print("Media KPI Summary:")
display(media_kpi)

# Total media investment over time
print("Overall Total Media Spend:", media['Total Investment'].sum())
print("Average Monthly Media Spend:", media['Total Investment'].mean())
print("Highest Monthly Total Investment:", media['Total Investment'].max())
print("Lowest Monthly Total Investment:", media['Total Investment'].min())

# Bar chart for total spend by channel
channel_totals.plot(kind='bar', figsize=(10,5))
plt.title("Total Spend by Media Channel")
plt.xlabel("Channel")
plt.ylabel("Total Spend")
plt.xticks(rotation=45)
plt.show()

In [ ]:
kpi_weekly = {}

# Revenue KPIs
for col in ['CameraAccessory', 'GamingAccessory', 'HomeAudio']:
    kpi_weekly[f'Total Revenue - {col}'] = weekly_pivot[col].sum()
    kpi_weekly[f'Average Weekly Revenue - {col}'] = weekly_pivot[col].mean()
    kpi_weekly[f'Median Weekly Revenue - {col}'] = weekly_pivot[col].median()
    kpi_weekly[f'Max Weekly Revenue - {col}'] = weekly_pivot[col].max()
    kpi_weekly[f'Min Weekly Revenue - {col}'] = weekly_pivot[col].min()

# Combined selected-category revenue
kpi_weekly['Total Revenue - All 3 Categories'] = (
    weekly_pivot['CameraAccessory'].sum() +
    weekly_pivot['GamingAccessory'].sum() +
    weekly_pivot['HomeAudio'].sum()
)

# Business KPIs
kpi_weekly['Number of Payday Weeks'] = weekly_pivot['is_payday'].sum()
kpi_weekly['Number of Holiday Weeks'] = weekly_pivot['is_holiday'].sum()

# Climate KPIs
kpi_weekly['Average Weekly Temperature'] = weekly_pivot['mean_temp_°c'].mean()
kpi_weekly['Total Rainfall'] = weekly_pivot['total_rain_mm'].sum()
kpi_weekly['Number of Rainy Weeks'] = weekly_pivot['is_rainy'].sum()
kpi_weekly['Number of Heavy Rain Weeks'] = weekly_pivot['heavy_rain'].sum()
kpi_weekly['Number of Extreme Temperature Weeks'] = weekly_pivot['extreme_temp'].sum()
kpi_weekly['Most Common Temperature Category'] = weekly_pivot['temp_category'].mode(dropna=True)[0]

kpi_weekly_df = pd.DataFrame(kpi_weekly.items(), columns=['KPI', 'Value'])
display(kpi_weekly_df)

Weekly dataset KPIs (weekly_pivot)
Revenue KPIs
Total Revenue - CameraAccessory
Total Revenue - GamingAccessory
Total Revenue - HomeAudio
Average Weekly Revenue - CameraAccessory
Average Weekly Revenue - GamingAccessory
Average Weekly Revenue - HomeAudio
Total Revenue - All 3 Categories
Business and climate KPIs
Number of Payday Weeks
Number of Rainy Weeks
Number of Heavy Rain Weeks
Number of Extreme Temperature Weeks
Average Weekly Temperature
Total Rainfall
Most Common Temperature Category
Do not highlight as useful KPI
Number of Holiday Weeks
Only mention it with a note that the holiday signal appears unreliable.
Media dataset KPIs (media)
Overall Total Media Spend
Average Monthly Media Spend
Highest Monthly Total Investment
Spend Share % by channel
Most important channel KPIs to highlight
Sponsorship Spend Share %
Online marketing Spend Share %
SEM Spend Share %

## MODEL BUILDING

In [ ]:
import pandas as pd
import numpy as np

# Make copies so original data stays safe
weekly_df = weekly_pivot.copy()
media_monthly = media.copy()
# -----------------------------
# Prepare weekly dataset
# -----------------------------
weekly_df['week_start'] = pd.to_datetime(weekly_df['week_start'])
weekly_df['week_end'] = weekly_df['week_start'] + pd.Timedelta(days=6)

print("Weekly dataset preview:")
print(weekly_df[['week_start', 'week_end']].head())

# -----------------------------
# Prepare monthly media dataset
# -----------------------------
media_monthly['Date'] = pd.to_datetime(media_monthly['Date'])

# Month start and month end
media_monthly['month_start'] = media_monthly['Date'].values.astype('datetime64[M]')
media_monthly['month_end'] = media_monthly['month_start'] + pd.offsets.MonthEnd(1)

print("\nMedia dataset preview:")
print(media_monthly[['Date', 'month_start', 'month_end']].head())

In [ ]:
media_cols = [
    'TV',
    'Digital',
    'Sponsorship',
    'Content Marketing',
    'Online marketing',
    ' Affiliates',
    'SEM'
]

# Convert to numeric just in case
for col in media_cols:
    media_monthly[col] = pd.to_numeric(media_monthly[col], errors='coerce')

print(media_monthly[media_cols].dtypes)
print("\nMissing values in media columns:")
print(media_monthly[media_cols].isnull().sum())

In [ ]:
weekly_media_rows = []

# keep unique weeks only
unique_weeks = weekly_df[['week_start', 'week_end']].drop_duplicates().sort_values('week_start')

for _, week_row in unique_weeks.iterrows():
    ws = week_row['week_start']
    we = week_row['week_end']

    # create one dictionary for this week
    week_alloc = {'week_start': ws}

    # initialize each media channel with 0
    for col in media_cols:
        week_alloc[col] = 0.0

    # loop through monthly media rows
    for _, month_row in media_monthly.iterrows():
        ms = month_row['month_start']
        me = month_row['month_end']

        # calculate overlap between week and month
        overlap_start = max(ws, ms)
        overlap_end = min(we, me)

        if overlap_start <= overlap_end:
            overlap_days = (overlap_end - overlap_start).days + 1
            days_in_month = (me - ms).days + 1

            # allocate spend proportionally
            for col in media_cols:
                week_alloc[col] += month_row[col] * (overlap_days / days_in_month)

    weekly_media_rows.append(week_alloc)

# create weekly media dataframe
weekly_media_df = pd.DataFrame(weekly_media_rows)

print("Weekly media shape:", weekly_media_df.shape)
print(weekly_media_df.head())

In [ ]:
weekly_media_check = weekly_media_df.copy()
weekly_media_check['month'] = weekly_media_check['week_start'].dt.to_period('M')

# sum weekly allocated values back to monthly
reconstructed_monthly = weekly_media_check.groupby('month')[media_cols].sum().reset_index()

# original monthly values
original_monthly = media_monthly.copy()
original_monthly['month'] = original_monthly['month_start'].dt.to_period('M')
original_monthly = original_monthly[['month'] + media_cols]

# compare both
validation_df = original_monthly.merge(
    reconstructed_monthly,
    on='month',
    suffixes=('_original', '_reconstructed')
)

print(validation_df.head())

In [ ]:
final_mmm_df = weekly_df.merge(
    weekly_media_df,
    on='week_start',
    how='left'
)

print("Final MMM dataset shape:", final_mmm_df.shape)
print(final_mmm_df.head())
print("\nMissing values in final dataset:")
print(final_mmm_df.isnull().sum())

In [ ]:
print("Columns in final_mmm_df:")
print(final_mmm_df.columns.tolist())

print("\nShape:")
print(final_mmm_df.shape)

print("\nMissing values:")
print(final_mmm_df.isnull().sum())

print("\nData types:")
print(final_mmm_df.dtypes)

In [ ]:
# Number of unique values in each column
unique_counts = final_mmm_df.nunique().sort_values()

print(unique_counts)

In [ ]:
model_df = final_mmm_df.copy()

drop_cols = ['week', 'week_start', 'week_end', 'day', 'is_holiday', 'temp_category']
drop_cols = [col for col in drop_cols if col in model_df.columns]

model_df = model_df.drop(columns=drop_cols)

print("Model dataset shape:", model_df.shape)
print(model_df.head())
print("\nRemaining columns:")
print(model_df.columns.tolist())

In [ ]:
print("Minimum values in model_df:")
print(model_df.min(numeric_only=True))

In [ ]:
print(final_mmm_df.columns.tolist())
print(final_mmm_df.nunique().sort_values())
print(model_df.columns.tolist())
print(model_df.min(numeric_only=True))

In [ ]:
print(final_mmm_df.isnull().sum()[final_mmm_df.isnull().sum() > 0])

In [ ]:
model_df = final_mmm_df.copy()

# Drop columns we decided not to use
drop_cols = ['week', 'week_start', 'week_end', 'day', 'is_holiday', 'temp_category']
drop_cols = [col for col in drop_cols if col in model_df.columns]
model_df = model_df.drop(columns=drop_cols)

# Drop rows with missing values
model_df = model_df.dropna().reset_index(drop=True)

print("Clean model dataset shape:", model_df.shape)
print(model_df.isnull().sum())
print(model_df.head())

In [ ]:
media_cols = [
    'TV',
    'Digital',
    'Sponsorship',
    'Content Marketing',
    'Online marketing',
    ' Affiliates',
    'SEM'
]

# make a copy
adstock_df = model_df.copy()

def adstock_transform(x, decay=0.5):
    result = []
    prev = 0
    for val in x:
        current = val + decay * prev
        result.append(current)
        prev = current
    return result

# create adstocked media columns
for col in media_cols:
    adstock_df[col + '_adstock'] = adstock_transform(adstock_df[col], decay=0.5)

print("Adstock dataset shape:", adstock_df.shape)
print(adstock_df[[col for col in adstock_df.columns if 'adstock' in col]].head())

In [ ]:
compare_cols = ['TV', 'TV_adstock', 'Digital', 'Digital_adstock', 'Sponsorship', 'Sponsorship_adstock']
print(adstock_df[compare_cols].head(10))

In [ ]:
target = 'CameraAccessory'

feature_cols = [
    'is_payday',
    'mean_temp_°c',
    'total_rain_mm',
    'is_rainy',
    'heavy_rain',
    'extreme_temp',
    'TV_adstock',
    'Digital_adstock',
    'Sponsorship_adstock',
    'Content Marketing_adstock',
    'Online marketing_adstock',
    ' Affiliates_adstock',
    'SEM_adstock'
]

X = adstock_df[feature_cols]
y = adstock_df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)
print(X.head())
print(y.head())

In [ ]:
split_index = int(len(adstock_df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Train size:", X_train.shape, y_train.shape)
print("Test size:", X_test.shape, y_test.shape)

In [ ]:
from sklearn.linear_model import LinearRegression

basic_model = LinearRegression()
basic_model.fit(X_train, y_train)

print("Model fitted successfully.")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_train_pred = basic_model.predict(X_train)
y_test_pred = basic_model.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print("Train R²:", train_r2)
print("Test R²:", test_r2)
print("Train RMSE:", train_rmse)
print("Test RMSE:", test_rmse)
print("Train MAE:", train_mae)
print("Test MAE:", test_mae)

In [ ]:
coef_df = pd.DataFrame({
    'Variable': feature_cols,
    'Coefficient': basic_model.coef_
}).sort_values(by='Coefficient', ascending=False)

print(coef_df)

In [ ]:
comparison_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_test_pred
})

print(comparison_df.head(10))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(y_test.values, label='Actual')
plt.plot(y_test_pred, label='Predicted')
plt.title('CameraAccessory: Actual vs Predicted (Basic Linear MMM)')
plt.xlabel('Test Weeks')
plt.ylabel('Revenue')
plt.legend()
plt.show()

In [ ]:
print("Train R²:", train_r2)
print("Test R²:", test_r2)
print("Train RMSE:", train_rmse)
print("Test RMSE:", test_rmse)
print(coef_df)
print(comparison_df.head(10))

In [ ]:
# log
import numpy as np

log_df = adstock_df.copy()

log_target = 'CameraAccessory'

log_feature_cols = [
    'is_payday',
    'mean_temp_°c',
    'total_rain_mm',
    'is_rainy',
    'heavy_rain',
    'extreme_temp',
    'TV_adstock',
    'Digital_adstock',
    'Sponsorship_adstock',
    'Content Marketing_adstock',
    'Online marketing_adstock',
    ' Affiliates_adstock',
    'SEM_adstock'
]

# create transformed copy
log_model_df = pd.DataFrame()

# target
log_model_df['log_target'] = np.log1p(log_df[log_target])

# predictors
for col in log_feature_cols:
    if col == 'mean_temp_°c':
        # temperature has negative values, so shift before log
        shift_val = abs(log_df[col].min()) + 1
        log_model_df[col + '_log'] = np.log(log_df[col] + shift_val)
    else:
        log_model_df[col + '_log'] = np.log1p(log_df[col])

print(log_model_df.head())
print(log_model_df.shape)

In [ ]:
X_log = log_model_df.drop(columns=['log_target'])
y_log = log_model_df['log_target']

split_index = int(len(log_model_df) * 0.8)

X_log_train = X_log.iloc[:split_index]
X_log_test = X_log.iloc[split_index:]

y_log_train = y_log.iloc[:split_index]
y_log_test = y_log.iloc[split_index:]

print("X_log_train:", X_log_train.shape)
print("X_log_test:", X_log_test.shape)
print("y_log_train:", y_log_train.shape)
print("y_log_test:", y_log_test.shape)

In [ ]:
from sklearn.linear_model import LinearRegression

log_model = LinearRegression()
log_model.fit(X_log_train, y_log_train)

print("Logarithmic model fitted successfully.")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_log_train_pred = log_model.predict(X_log_train)
y_log_test_pred = log_model.predict(X_log_test)

log_train_r2 = r2_score(y_log_train, y_log_train_pred)
log_test_r2 = r2_score(y_log_test, y_log_test_pred)

log_train_rmse = np.sqrt(mean_squared_error(y_log_train, y_log_train_pred))
log_test_rmse = np.sqrt(mean_squared_error(y_log_test, y_log_test_pred))

log_train_mae = mean_absolute_error(y_log_train, y_log_train_pred)
log_test_mae = mean_absolute_error(y_log_test, y_log_test_pred)

print("Log Train R²:", log_train_r2)
print("Log Test R²:", log_test_r2)
print("Log Train RMSE:", log_train_rmse)
print("Log Test RMSE:", log_test_rmse)
print("Log Train MAE:", log_train_mae)
print("Log Test MAE:", log_test_mae)

In [ ]:
log_coef_df = pd.DataFrame({
    'Variable': X_log.columns,
    'Coefficient': log_model.coef_
}).sort_values(by='Coefficient', ascending=False)

print(log_coef_df)

In [ ]:
print("Log Train R²:", log_train_r2)
print("Log Test R²:", log_test_r2)
print("Log Train RMSE:", log_train_rmse)
print("Log Test RMSE:", log_test_rmse)
print(log_coef_df)

In [ ]:
mult_df = adstock_df.copy()

# Target
mult_target = 'CameraAccessory'

# Continuous predictors only
mult_features = [
    'mean_temp_°c',
    'total_rain_mm',
    'TV_adstock',
    'Digital_adstock',
    'Sponsorship_adstock',
    'Content Marketing_adstock',
    'Online marketing_adstock',
    'Affiliates_adstock',
    'SEM_adstock'
]

mult_model_df = pd.DataFrame()

# log target
mult_model_df['log_target'] = np.log(mult_df[mult_target])

# shift temperature because it has negative values
temp_shift = abs(mult_df['mean_temp_°c'].min()) + 1
mult_model_df['mean_temp_°c_log'] = np.log(mult_df['mean_temp_°c'] + temp_shift)

# rainfall can be zero, so use log1p
mult_model_df['total_rain_mm_log'] = np.log1p(mult_df['total_rain_mm'])

# media variables are positive, so log directly
for col in [
    'TV_adstock',
    'Digital_adstock',
    'Sponsorship_adstock',
    'Content Marketing_adstock',
    'Online marketing_adstock',
    ' Affiliates_adstock',
    'SEM_adstock'
]:
    mult_model_df[col + '_log'] = np.log(mult_df[col])

print(mult_model_df.head())
print(mult_model_df.shape)

In [ ]:
X_mult = mult_model_df.drop(columns=['log_target'])
y_mult = mult_model_df['log_target']

split_index = int(len(mult_model_df) * 0.8)

X_mult_train = X_mult.iloc[:split_index]
X_mult_test = X_mult.iloc[split_index:]

y_mult_train = y_mult.iloc[:split_index]
y_mult_test = y_mult.iloc[split_index:]

print("X_mult_train:", X_mult_train.shape)
print("X_mult_test:", X_mult_test.shape)
print("y_mult_train:", y_mult_train.shape)
print("y_mult_test:", y_mult_test.shape)

In [ ]:
mult_model = LinearRegression()
mult_model.fit(X_mult_train, y_mult_train)

print("Multiplicative model fitted successfully.")

In [ ]:
y_mult_train_pred = mult_model.predict(X_mult_train)
y_mult_test_pred = mult_model.predict(X_mult_test)

mult_train_r2 = r2_score(y_mult_train, y_mult_train_pred)
mult_test_r2 = r2_score(y_mult_test, y_mult_test_pred)

mult_train_rmse = np.sqrt(mean_squared_error(y_mult_train, y_mult_train_pred))
mult_test_rmse = np.sqrt(mean_squared_error(y_mult_test, y_mult_test_pred))

mult_train_mae = mean_absolute_error(y_mult_train, y_mult_train_pred)
mult_test_mae = mean_absolute_error(y_mult_test, y_mult_test_pred)

print("Multiplicative Train R²:", mult_train_r2)
print("Multiplicative Test R²:", mult_test_r2)
print("Multiplicative Train RMSE:", mult_train_rmse)
print("Multiplicative Test RMSE:", mult_test_rmse)
print("Multiplicative Train MAE:", mult_train_mae)
print("Multiplicative Test MAE:", mult_test_mae)

In [ ]:
mult_coef_df = pd.DataFrame({
    'Variable': X_mult.columns,
    'Coefficient': mult_model.coef_
}).sort_values(by='Coefficient', ascending=False)

print(mult_coef_df)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

def run_mmm_models_for_target(adstock_df, target_col):
    results = []

    # -------------------------
    # 1. BASIC MODEL
    # -------------------------
    basic_features = [
        'is_payday',
        'mean_temp_°c',
        'total_rain_mm',
        'is_rainy',
        'heavy_rain',
        'extreme_temp',
        'TV_adstock',
        'Digital_adstock',
        'Sponsorship_adstock',
        'Content Marketing_adstock',
        'Online marketing_adstock',
        ' Affiliates_adstock',
        'SEM_adstock'
    ]

    X_basic = adstock_df[basic_features]
    y_basic = adstock_df[target_col]

    split_index = int(len(adstock_df) * 0.8)

    Xb_train = X_basic.iloc[:split_index]
    Xb_test = X_basic.iloc[split_index:]
    yb_train = y_basic.iloc[:split_index]
    yb_test = y_basic.iloc[split_index:]

    basic_model = LinearRegression()
    basic_model.fit(Xb_train, yb_train)

    yb_train_pred = basic_model.predict(Xb_train)
    yb_test_pred = basic_model.predict(Xb_test)

    results.append({
        'Target': target_col,
        'Model': 'Basic Linear',
        'Train_R2': r2_score(yb_train, yb_train_pred),
        'Test_R2': r2_score(yb_test, yb_test_pred),
        'Train_RMSE': np.sqrt(mean_squared_error(yb_train, yb_train_pred)),
        'Test_RMSE': np.sqrt(mean_squared_error(yb_test, yb_test_pred)),
        'Train_MAE': mean_absolute_error(yb_train, yb_train_pred),
        'Test_MAE': mean_absolute_error(yb_test, yb_test_pred)
    })

    # -------------------------
    # 2. LOG MODEL
    # -------------------------
    log_df = pd.DataFrame()
    log_df['log_target'] = np.log1p(adstock_df[target_col])

    for col in basic_features:
        if col == 'mean_temp_°c':
            shift_val = abs(adstock_df[col].min()) + 1
            log_df[col + '_log'] = np.log(adstock_df[col] + shift_val)
        else:
            log_df[col + '_log'] = np.log1p(adstock_df[col])

    X_log = log_df.drop(columns=['log_target'])
    y_log = log_df['log_target']

    Xl_train = X_log.iloc[:split_index]
    Xl_test = X_log.iloc[split_index:]
    yl_train = y_log.iloc[:split_index]
    yl_test = y_log.iloc[split_index:]

    log_model = LinearRegression()
    log_model.fit(Xl_train, yl_train)

    yl_train_pred = log_model.predict(Xl_train)
    yl_test_pred = log_model.predict(Xl_test)

    results.append({
        'Target': target_col,
        'Model': 'Logarithmic',
        'Train_R2': r2_score(yl_train, yl_train_pred),
        'Test_R2': r2_score(yl_test, yl_test_pred),
        'Train_RMSE': np.sqrt(mean_squared_error(yl_train, yl_train_pred)),
        'Test_RMSE': np.sqrt(mean_squared_error(yl_test, yl_test_pred)),
        'Train_MAE': mean_absolute_error(yl_train, yl_train_pred),
        'Test_MAE': mean_absolute_error(yl_test, yl_test_pred)
    })

    # -------------------------
    # 3. MULTIPLICATIVE MODEL
    # -------------------------
    mult_df = pd.DataFrame()
    mult_df['log_target'] = np.log(adstock_df[target_col])

    temp_shift = abs(adstock_df['mean_temp_°c'].min()) + 1
    mult_df['mean_temp_°c_log'] = np.log(adstock_df['mean_temp_°c'] + temp_shift)
    mult_df['total_rain_mm_log'] = np.log1p(adstock_df['total_rain_mm'])

    mult_media = [
        'TV_adstock',
        'Digital_adstock',
        'Sponsorship_adstock',
        'Content Marketing_adstock',
        'Online marketing_adstock',
        ' Affiliates_adstock',
        'SEM_adstock'
    ]

    for col in mult_media:
        mult_df[col + '_log'] = np.log(adstock_df[col])

    X_mult = mult_df.drop(columns=['log_target'])
    y_mult = mult_df['log_target']

    Xm_train = X_mult.iloc[:split_index]
    Xm_test = X_mult.iloc[split_index:]
    ym_train = y_mult.iloc[:split_index]
    ym_test = y_mult.iloc[split_index:]

    mult_model = LinearRegression()
    mult_model.fit(Xm_train, ym_train)

    ym_train_pred = mult_model.predict(Xm_train)
    ym_test_pred = mult_model.predict(Xm_test)

    results.append({
        'Target': target_col,
        'Model': 'Multiplicative',
        'Train_R2': r2_score(ym_train, ym_train_pred),
        'Test_R2': r2_score(ym_test, ym_test_pred),
        'Train_RMSE': np.sqrt(mean_squared_error(ym_train, ym_train_pred)),
        'Test_RMSE': np.sqrt(mean_squared_error(ym_test, ym_test_pred)),
        'Train_MAE': mean_absolute_error(ym_train, ym_train_pred),
        'Test_MAE': mean_absolute_error(ym_test, ym_test_pred)
    })

    return pd.DataFrame(results)

In [ ]:
gaming_results = run_mmm_models_for_target(adstock_df, 'GamingAccessory')
homeaudio_results = run_mmm_models_for_target(adstock_df, 'HomeAudio')

print("GamingAccessory Results")
print(gaming_results)

print("\nHomeAudio Results")
print(homeaudio_results)

In [ ]:
camera_results = pd.DataFrame([
    {
        'Target': 'CameraAccessory',
        'Model': 'Basic Linear',
        'Train_R2': train_r2,
        'Test_R2': test_r2,
        'Train_RMSE': train_rmse,
        'Test_RMSE': test_rmse,
        'Train_MAE': train_mae,
        'Test_MAE': test_mae
    },
    {
        'Target': 'CameraAccessory',
        'Model': 'Logarithmic',
        'Train_R2': log_train_r2,
        'Test_R2': log_test_r2,
        'Train_RMSE': log_train_rmse,
        'Test_RMSE': log_test_rmse,
        'Train_MAE': log_train_mae,
        'Test_MAE': log_test_mae
    },
    {
        'Target': 'CameraAccessory',
        'Model': 'Multiplicative',
        'Train_R2': mult_train_r2,
        'Test_R2': mult_test_r2,
        'Train_RMSE': mult_train_rmse,
        'Test_RMSE': mult_test_rmse,
        'Train_MAE': mult_train_mae,
        'Test_MAE': mult_test_mae
    }
])

all_results = pd.concat([camera_results, gaming_results, homeaudio_results], ignore_index=True)

print(all_results)

In [ ]:
final_selection = pd.DataFrame({
    'Category': ['CameraAccessory', 'GamingAccessory', 'HomeAudio'],
    'Selected_Model': ['Basic Linear', 'Basic Linear', 'Logarithmic'],
    'Best_Test_R2': [-0.422381, -0.023041, -3.286245]
})

print(final_selection)

##INSIGHTS AND RECOMMENDATIONS

Insights

The Market Mix Modeling exercise was completed for three product categories: CameraAccessory, GamingAccessory, and HomeAudio. For each category, three model forms were tested: basic linear, logarithmic, and multiplicative. Model selection was based primarily on out-of-sample performance using test R².

For CameraAccessory, the basic linear model performed better than the logarithmic and multiplicative models, with the highest test R² among the three alternatives. Although the predictive strength remained limited, the linear form was relatively more suitable for this category.

For GamingAccessory, the basic linear model again emerged as the best-performing specification. It also produced the best relative test performance across all three categories, suggesting that GamingAccessory revenue is comparatively more stable and predictable than the other categories.

For HomeAudio, the logarithmic model performed better than the basic linear and multiplicative specifications. This suggests that HomeAudio revenue may respond more appropriately to proportional or non-linear effects rather than purely linear changes in the predictors.

Overall, none of the models achieved strong positive out-of-sample explanatory power. This indicates that while the modeling workflow was correct, the current data may not fully capture all important drivers of weekly revenue.

Recommendations

The results suggest that future MMM improvement would benefit from a larger historical dataset, because the current weekly sample is relatively small for the number of predictors used. More observations would likely improve model stability and generalization.

It is also recommended to include additional explanatory variables such as promotions, pricing changes, competitor actions, seasonality indicators, and major business events, since these may explain sales fluctuations not captured by media and climate variables alone.

Because several media channels may move together, multicollinearity is likely affecting coefficient stability. Future work could reduce this issue by combining similar channels, selecting fewer media variables, or applying regularized techniques.

From a business perspective, GamingAccessory appears to be the most stable category, while CameraAccessory and HomeAudio show greater volatility. This means forecasting and media impact measurement may be easier for GamingAccessory than for the other two categories.

In conclusion, the project successfully implemented the full MMM workflow, including media alignment, adstock transformation, model building, and comparative evaluation. Even though predictive performance was limited, the analysis provided a structured and evidence-based understanding of how different model forms behave across product categories.

Insights

The Market Mix Modeling exercise was completed for three product categories: CameraAccessory, GamingAccessory, and HomeAudio. For each category, three model forms were tested: basic linear, logarithmic, and multiplicative. Model selection was based primarily on out-of-sample performance using test R².

For CameraAccessory, the basic linear model performed better than the logarithmic and multiplicative models, with the highest test R² among the three alternatives. Although the predictive strength remained limited, the linear form was relatively more suitable for this category.

For GamingAccessory, the basic linear model again emerged as the best-performing specification. It also produced the best relative test performance across all three categories, suggesting that GamingAccessory revenue is comparatively more stable and predictable than the other categories.

For HomeAudio, the logarithmic model performed better than the basic linear and multiplicative specifications. This suggests that HomeAudio revenue may respond more appropriately to proportional or non-linear effects rather than purely linear changes in the predictors.

Overall, none of the models achieved strong positive out-of-sample explanatory power. This indicates that while the modeling workflow was correct, the current data may not fully capture all important drivers of weekly revenue.

Recommendations

The results suggest that future MMM improvement would benefit from a larger historical dataset, because the current weekly sample is relatively small for the number of predictors used. More observations would likely improve model stability and generalization.

It is also recommended to include additional explanatory variables such as promotions, pricing changes, competitor actions, seasonality indicators, and major business events, since these may explain sales fluctuations not captured by media and climate variables alone.

Because several media channels may move together, multicollinearity is likely affecting coefficient stability. Future work could reduce this issue by combining similar channels, selecting fewer media variables, or applying regularized techniques.

From a business perspective, GamingAccessory appears to be the most stable category, while CameraAccessory and HomeAudio show greater volatility. This means forecasting and media impact measurement may be easier for GamingAccessory than for the other two categories.

In conclusion, the project successfully implemented the full MMM workflow, including media alignment, adstock transformation, model building, and comparative evaluation. Even though predictive performance was limited, the analysis provided a structured and evidence-based understanding of how different model forms behave across product categories.
